# 4.2 怎麼保留原本的資訊？

# 第 4 章：第一個 Dense Transformer 語言模型

前置：第3章的因果 attention。完整可讀實作在 tiny_perceptron/model.py；預設只用 learned position、LayerNorm、Dense FFN，還沒有 FlashAttention 或 MoE。




In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("先依 course/README.md 下載並開啟這個 repo")
sys.path.insert(0, str(root))
import torch
from torch import nn
from torch.nn import functional as F

torch.set_num_threads(1)
torch.manual_seed(42)

**先看圖：像原稿旁加修訂，而不是撕掉原稿**

y=x+f(x)，原資料和修訂都有路徑。

橘色邊框只提示閱讀順序，靜態標註一直保留。系統的減少動態效果設定會停用動畫。


In [ ]:
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/residual.svg")))

**先想一想：**f(x)=2x 時，梯度是2還是3？

讓一條路直接保留原資料，另一條路只補上修正。即使修正暫時很小，原資訊仍能向後走，梯度也有直接路徑。

**把直覺寫成數學：**$y=x+f(x)$，$dy/dx=I+df/dx$。

**最小實驗：**以下程式只處理這節的問題。


In [ ]:
x = torch.tensor([1.0, 2.0], requires_grad=True)
y = x + 2 * x
y.sum().backward()
print(y.detach(), x.grad)
assert torch.equal(x.grad, torch.tensor([3.0, 3.0]))

**如何讀結果：**residual 不是把兩個 loss 相加，而是同一個表示的兩條路合起來。

**只改一件事：**只把 f(x) 改成0*x。
